# WLED UDP Realtime Test

This notebook tests UDP connectivity to a WLED controller using the DRGB (Realtime) protocol.

**WLED Settings Required:**
- UDP Realtime must be enabled in Settings → Sync Interfaces
- Default UDP port: 21324
- LED count should match your strip (default: 180)

In [1]:
import socket
import time
import struct
import requests
from typing import Optional, Tuple

In [14]:
# WLED Configuration
WLED_HOST = "10.0.0.43"
WLED_UDP_PORT = 21324
WLED_HTTP_PORT = 80
LED_COUNT = 180  # Number of LEDs in your strip

print(f"WLED Controller: {WLED_HOST}:{WLED_UDP_PORT}")
print(f"LED Count: {LED_COUNT}")
print(f"HTTP API: http://{WLED_HOST}:{WLED_HTTP_PORT}/json")

WLED Controller: 10.0.0.43:21324
LED Count: 180
HTTP API: http://10.0.0.43:80/json


In [15]:
# Check if this is a GRB LED strip
IS_GRB_STRIP = False  # CHANGED: Set to False since colors appeared swapped

print(f"LED Strip Type: {'GRB' if IS_GRB_STRIP else 'RGB'}")
print(f"WLED Controller: {WLED_HOST}:{WLED_UDP_PORT}")
print(f"LED Count: {LED_COUNT}")
print(f"HTTP API: http://{WLED_HOST}:{WLED_HTTP_PORT}/json")

LED Strip Type: RGB
WLED Controller: 10.0.0.43:21324
LED Count: 180
HTTP API: http://10.0.0.43:80/json


In [11]:
class WLEDUDPClient:
    """UDP client for WLED DRGB (Realtime) protocol"""

    def __init__(self, host: str, port: int, led_count: int, is_grb: bool = False):
        self.host = host
        self.port = port
        self.led_count = led_count
        self.is_grb = is_grb  # True for GRB strips, False for RGB
        self.sock: Optional[socket.socket] = None

    def connect(self) -> bool:
        """Create and bind UDP socket"""
        try:
            self.sock = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
            self.sock.settimeout(1.0)  # 1 second timeout
            print(f"✓ UDP socket created for {self.host}:{self.port}")
            return True
        except Exception as e:
            print(f"✗ Failed to create UDP socket: {e}")
            return False

    def disconnect(self):
        """Close UDP socket"""
        if self.sock:
            self.sock.close()
            self.sock = None
            print("✓ UDP socket closed")

    def send_frame(self, rgb_data: bytes, timeout: int = 1) -> bool:
        """Send DRGB protocol frame to WLED

        DRGB Protocol:
        - Byte 0: Protocol ID (0x01 for DRGB)
        - Byte 1: Timeout in seconds (0 = live mode)
        - Bytes 2+: RGB data (3 bytes per LED)
        """
        if not self.sock:
            print("✗ No UDP socket available")
            return False

        try:
            # Build DRGB packet
            packet = bytearray()
            packet.append(0x02)  # DRGB protocol (corrected from 0x01)
            packet.append(timeout)  # Timeout in seconds
            packet.extend(rgb_data)  # RGB data

            # Send packet
            self.sock.sendto(packet, (self.host, self.port))
            print(f"✓ Sent {len(packet)} bytes to {self.host}:{self.port}")
            return True

        except Exception as e:
            print(f"✗ Failed to send UDP packet: {e}")
            return False

    def create_solid_color_frame(self, r: int, g: int, b: int) -> bytes:
        """Create RGB/GRB data for solid color across all LEDs"""
        rgb_data = bytearray()
        for _ in range(self.led_count):
            if self.is_grb:
                # GRB order: Green, Red, Blue
                rgb_data.extend([g, r, b])
            else:
                # RGB order: Red, Green, Blue
                rgb_data.extend([r, g, b])
        return bytes(rgb_data)

# Create WLED client
wled = WLEDUDPClient(WLED_HOST, WLED_UDP_PORT, LED_COUNT, IS_GRB_STRIP)

## Test 1: Basic Network Connectivity

First, let's test if we can reach the WLED controller on the network.

In [5]:
def test_network_connectivity(host: str, port: int) -> Tuple[bool, str]:
    """Test basic network connectivity to host:port"""
    try:
        # Test TCP connection (HTTP port)
        tcp_sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        tcp_sock.settimeout(5.0)
        result = tcp_sock.connect_ex((host, port))
        tcp_sock.close()

        if result == 0:
            return True, f"✓ TCP connection to {host}:{port} successful"
        else:
            return False, f"✗ TCP connection to {host}:{port} failed (error code: {result})"

    except Exception as e:
        return False, f"✗ Network test failed: {e}"

# Test connectivity
print("Testing network connectivity...")
tcp_ok, tcp_msg = test_network_connectivity(WLED_HOST, WLED_HTTP_PORT)
print(tcp_msg)

# Test UDP port (this will always succeed if network is reachable)
udp_ok, udp_msg = test_network_connectivity(WLED_HOST, WLED_UDP_PORT)
print("UDP port test: Network reachable" if tcp_ok else "UDP port test: Network unreachable")

Testing network connectivity...
✓ TCP connection to 10.0.0.43:80 successful
UDP port test: Network reachable


## Test 2: WLED HTTP API Status

Let's check if WLED is responding to HTTP requests and get its current status.

In [6]:
def test_wled_api(host: str, port: int) -> Tuple[bool, dict]:
    """Test WLED HTTP API and get status"""
    try:
        url = f"http://{host}:{port}/json"
        response = requests.get(url, timeout=5)

        if response.status_code == 200:
            data = response.json()
            return True, {
                "name": data.get("name", "Unknown"),
                "version": data.get("ver", "Unknown"),
                "leds": data.get("leds", {}),
                "state": data.get("state", {}),
                "effects": data.get("effects", [])
            }
        else:
            return False, {"error": f"HTTP {response.status_code}"}

    except Exception as e:
        return False, {"error": str(e)}

# Test WLED API
print("Testing WLED HTTP API...")
api_ok, api_data = test_wled_api(WLED_HOST, WLED_HTTP_PORT)

if api_ok:
    print("✓ WLED API responding")
    print(f"  Name: {api_data['name']}")
    print(f"  Version: {api_data['version']}")
    print(f"  LED Count: {api_data['leds'].get('count', 'Unknown')}")
    print(f"  Current Effect: {api_data['state'].get('seg', [{}])[0].get('fx', 'Unknown')}")
else:
    print(f"✗ WLED API not responding: {api_data.get('error', 'Unknown error')}")

api_data

Testing WLED HTTP API...
✓ WLED API responding
  Name: Unknown
  Version: Unknown
  LED Count: Unknown
  Current Effect: 2


{'name': 'Unknown',
 'version': 'Unknown',
 'leds': {},
 'state': {'on': True,
  'bri': 50,
  'transition': 3,
  'ps': -1,
  'pl': -1,
  'ledmap': 0,
  'nl': {'on': False, 'dur': 60, 'mode': 1, 'tbri': 0, 'rem': -1},
  'udpn': {'send': False, 'recv': True, 'sgrp': 1, 'rgrp': 1},
  'lor': 0,
  'mainseg': 1,
  'seg': [{'id': 0,
    'start': 0,
    'stop': 180,
    'len': 180,
    'grp': 1,
    'spc': 0,
    'of': 0,
    'on': True,
    'frz': False,
    'bri': 255,
    'cct': 127,
    'set': 0,
    'col': [[255, 0, 10], [0, 0, 0], [0, 0, 0]],
    'fx': 2,
    'sx': 10,
    'ix': 128,
    'pal': 8,
    'c1': 128,
    'c2': 128,
    'c3': 16,
    'sel': False,
    'rev': False,
    'mi': False,
    'o1': False,
    'o2': False,
    'o3': False,
    'si': 0,
    'm12': 0},
   {'id': 1,
    'start': 0,
    'stop': 180,
    'len': 180,
    'grp': 1,
    'spc': 0,
    'of': 0,
    'on': True,
    'frz': False,
    'bri': 82,
    'cct': 127,
    'set': 0,
    'col': [[0, 0, 200], [0, 0, 0], [0,

In [7]:
if api_ok:
    print("✓ WLED API responding")
    print(f"  Name: {api_data['name']}")
    print(f"  Version: {api_data['version']}")
    
    # Get LED count from segment (not from leds.count which may be empty)
    segments = api_data['state'].get('seg', [])
    if segments:
        wled_led_count = segments[0].get('len', 0)
    else:
        wled_led_count = 0
        
    print(f"  LED Count: {wled_led_count}")
    print(f"  Current Effect: {api_data['state'].get('seg', [{}])[0].get('fx', 'Unknown')}")

    # Check if LED count matches
    if wled_led_count != LED_COUNT:
        print(f"⚠️  LED count mismatch! WLED reports {wled_led_count} LEDs, but we're configured for {LED_COUNT}")
        print("   Update LED_COUNT variable above to match WLED's configuration")
else:
    print(f"✗ WLED API not responding: {api_data.get('error', 'Unknown error')}")

✓ WLED API responding
  Name: Unknown
  Version: Unknown
  LED Count: 180
  Current Effect: 2


In [8]:
# Debug: Show full WLED configuration
if api_ok:
    print("\n--- WLED Configuration Details ---")
    print(f"LED Configuration: {api_data['leds']}")

    # Check segments
    segments = api_data['state'].get('seg', [])
    if segments:
        seg0 = segments[0]
        print(f"Segment 0: start={seg0.get('start', 'N/A')}, stop={seg0.get('stop', 'N/A')}, len={seg0.get('len', 'N/A')}")
        print(f"Segment 0 effect: {seg0.get('fx', 'N/A')} (ID: {seg0.get('id', 'N/A')})")

    # Check realtime settings
    realtime = api_data['state'].get('udpn', {})
    print(f"UDP Realtime enabled: {realtime.get('send', False)}")
    print(f"UDP Realtime receive: {realtime.get('recv', False)}")

    # Calculate expected packet size
    expected_leds = wled_led_count  # Use the corrected LED count
    expected_bytes = expected_leds * 3 + 2  # 3 bytes per LED + 2 header bytes
    print(f"Expected packet size: {expected_bytes} bytes ({expected_leds} LEDs)")

    if expected_leds != LED_COUNT:
        print(f"⚠️  CRITICAL: LED count mismatch!")
        print(f"   WLED configured for: {expected_leds} LEDs")
        print(f"   Notebook configured for: {LED_COUNT} LEDs")
        print(f"   Update LED_COUNT = {expected_leds} in the configuration cell above")
else:
    print("Cannot check WLED configuration - API not responding")


--- WLED Configuration Details ---
LED Configuration: {}
Segment 0: start=0, stop=180, len=180
Segment 0 effect: 2 (ID: 0)
UDP Realtime enabled: False
UDP Realtime receive: True
Expected packet size: 542 bytes (180 LEDs)


## Test 3: UDP DRGB Protocol Test

Now let's test the actual UDP Realtime functionality. This sends DRGB protocol packets to control the LEDs directly.

**Important:** Make sure "Receive UDP realtime" is enabled in WLED Settings → Sync Interfaces.

In [16]:
def test_udp_colors():
    """Test UDP DRGB protocol with different colors"""
    print("Testing UDP DRGB protocol...")

    # Connect
    if not wled.connect():
        print("✗ Failed to create UDP socket")
        return False

    try:
        # Test colors
        colors = [
            ("RED", 255, 0, 0),
            ("GREEN", 0, 255, 0),
            ("BLUE", 0, 0, 255),
            ("WHITE", 255, 255, 255),
            ("OFF", 0, 0, 0)
        ]

        for name, r, g, b in colors:
            print(f"\n--- Testing {name} ---")

            # Create frame data
            frame_data = wled.create_solid_color_frame(r, g, b)
            print(f"Created frame: {len(frame_data)} bytes ({len(frame_data)//3} LEDs)")

            # Send frame with 2-second timeout
            if wled.send_frame(frame_data, 2):
                print(f"✓ {name} frame sent successfully")
                print(f"  Check your LED strip - it should show solid {name.lower()}")
            else:
                print(f"✗ Failed to send {name} frame")
                return False

            # Wait to see the color
            if name != "OFF":
                print("  Waiting 3 seconds to view color...")
                time.sleep(3)
            else:
                print("  Turning off LEDs...")
                time.sleep(1)

        print("\n✓ UDP DRGB test completed successfully!")
        return True

    finally:
        wled.disconnect()

# Run the UDP test
udp_test_result = test_udp_colors()

Testing UDP DRGB protocol...
✓ UDP socket created for 10.0.0.43:21324

--- Testing RED ---
Created frame: 540 bytes (180 LEDs)
✓ Sent 542 bytes to 10.0.0.43:21324
✓ RED frame sent successfully
  Check your LED strip - it should show solid red
  Waiting 3 seconds to view color...

--- Testing GREEN ---
Created frame: 540 bytes (180 LEDs)
✓ Sent 542 bytes to 10.0.0.43:21324
✓ GREEN frame sent successfully
  Check your LED strip - it should show solid green
  Waiting 3 seconds to view color...

--- Testing GREEN ---
Created frame: 540 bytes (180 LEDs)
✓ Sent 542 bytes to 10.0.0.43:21324
✓ GREEN frame sent successfully
  Check your LED strip - it should show solid green
  Waiting 3 seconds to view color...

--- Testing BLUE ---
Created frame: 540 bytes (180 LEDs)
✓ Sent 542 bytes to 10.0.0.43:21324
✓ BLUE frame sent successfully
  Check your LED strip - it should show solid blue
  Waiting 3 seconds to view color...

--- Testing BLUE ---
Created frame: 540 bytes (180 LEDs)
✓ Sent 542 bytes 

In [13]:
def test_single_led():
    """Test just the first LED to isolate the issue"""
    print("Testing single LED (first LED only)...")

    # Connect
    if not wled.connect():
        print("✗ Failed to create UDP socket")
        return False

    try:
        # Test single LED colors
        colors = [
            ("RED", 255, 0, 0),
            ("GREEN", 0, 255, 0),
            ("BLUE", 0, 0, 255),
            ("WHITE", 255, 255, 255),
            ("OFF", 0, 0, 0)
        ]

        for name, r, g, b in colors:
            print(f"\n--- Testing {name} (first LED only) ---")

            # Create frame data for just 1 LED
            if wled.is_grb:
                frame_data = bytes([g, r, b])  # GRB order
            else:
                frame_data = bytes([r, g, b])  # RGB order

            print(f"Created frame: {len(frame_data)} bytes (1 LED)")

            # Send frame with 3-second timeout
            if wled.send_frame(frame_data, 3):
                print(f"✓ {name} frame sent successfully")
                print(f"  Check your first LED - it should show {name.lower()}")
            else:
                print(f"✗ Failed to send {name} frame")
                return False

            # Wait to see the color
            print("  Waiting 2 seconds to view color...")
            time.sleep(2)

        print("\n✓ Single LED test completed!")
        return True

    finally:
        wled.disconnect()

# Test single LED first
print("\n" + "="*50)
print("FIRST: Testing single LED to isolate the issue")
print("="*50)
single_led_result = test_single_led()

# Then test all LEDs
print("\n" + "="*50)
print("SECOND: Testing all LEDs")
print("="*50)
udp_test_result = test_udp_colors()


FIRST: Testing single LED to isolate the issue
Testing single LED (first LED only)...
✓ UDP socket created for 10.0.0.43:21324

--- Testing RED (first LED only) ---
Created frame: 3 bytes (1 LED)
✓ Sent 5 bytes to 10.0.0.43:21324
✓ RED frame sent successfully
  Check your first LED - it should show red
  Waiting 2 seconds to view color...

--- Testing GREEN (first LED only) ---
Created frame: 3 bytes (1 LED)
✓ Sent 5 bytes to 10.0.0.43:21324
✓ GREEN frame sent successfully
  Check your first LED - it should show green
  Waiting 2 seconds to view color...

--- Testing BLUE (first LED only) ---
Created frame: 3 bytes (1 LED)
✓ Sent 5 bytes to 10.0.0.43:21324
✓ BLUE frame sent successfully
  Check your first LED - it should show blue
  Waiting 2 seconds to view color...

--- Testing WHITE (first LED only) ---
Created frame: 3 bytes (1 LED)
✓ Sent 5 bytes to 10.0.0.43:21324
✓ WHITE frame sent successfully
  Check your first LED - it should show white
  Waiting 2 seconds to view color...



In [ ]:
def test_led_count():
    """Test to determine actual LED count by lighting up groups"""
    print("Testing LED count by lighting up groups...")

    # Connect
    if not wled.connect():
        print("✗ Failed to create UDP socket")
        return False

    try:
        # Test different LED counts
        test_counts = [1, 5, 10, 30, 60, 120, 180]

        for count in test_counts:
            print(f"\n--- Testing {count} LEDs ---")

            # Create frame data for count LEDs (first N LEDs red, rest off)
            rgb_data = bytearray()
            for i in range(wled.led_count):
                if i < count:
                    if wled.is_grb:
                        rgb_data.extend([0, 255, 0])  # Red in GRB = G=0,R=255,B=0
                    else:
                        rgb_data.extend([255, 0, 0])  # Red in RGB
                else:
                    rgb_data.extend([0, 0, 0])  # Off

            print(f"Created frame: {len(rgb_data)} bytes ({len(rgb_data)//3} LEDs)")

            # Send frame
            if wled.send_frame(bytes(rgb_data), 3):
                print(f"✓ Sent frame for {count} LEDs")
                print(f"  You should see the first {count} LEDs light up RED")
            else:
                print(f"✗ Failed to send frame for {count} LEDs")
                return False

            print("  Waiting 3 seconds...")
            time.sleep(3)

        # Turn all off
        off_data = bytes([0, 0, 0] * wled.led_count)
        wled.send_frame(off_data, 1)
        print("✓ Turned all LEDs off")

        print("\n✓ LED count test completed!")
        return True

    finally:
        wled.disconnect()

# Test LED count
print("\n" + "="*50)
print("THIRD: Testing LED count with groups")
print("="*50)
led_count_result = test_led_count()


THIRD: Testing LED count with groups
Testing LED count by lighting up groups...
✓ UDP socket created for 10.0.0.43:21324

--- Testing 1 LEDs ---
Created frame: 540 bytes (180 LEDs)
✓ Sent 542 bytes to 10.0.0.43:21324
✓ Sent frame for 1 LEDs
  You should see the first 1 LEDs light up RED
  Waiting 3 seconds...

--- Testing 5 LEDs ---
Created frame: 540 bytes (180 LEDs)
✓ Sent 542 bytes to 10.0.0.43:21324
✓ Sent frame for 5 LEDs
  You should see the first 5 LEDs light up RED
  Waiting 3 seconds...

--- Testing 10 LEDs ---
Created frame: 540 bytes (180 LEDs)
✓ Sent 542 bytes to 10.0.0.43:21324
✓ Sent frame for 10 LEDs
  You should see the first 10 LEDs light up RED
  Waiting 3 seconds...

--- Testing 30 LEDs ---
Created frame: 540 bytes (180 LEDs)
✓ Sent 542 bytes to 10.0.0.43:21324
✓ Sent frame for 30 LEDs
  You should see the first 30 LEDs light up RED
  Waiting 3 seconds...

--- Testing 60 LEDs ---
Created frame: 540 bytes (180 LEDs)
✓ Sent 542 bytes to 10.0.0.43:21324
✓ Sent frame fo

KeyboardInterrupt: 

## Test Results Summary

✅ **WLED Configuration Verified:**
- LED Count: 180 (matches notebook configuration)
- UDP Realtime: Enabled (Receive UDP realtime = True)
- Segment 0: Configured for 180 LEDs (start=0, stop=180, len=180)
- Expected packet size: 542 bytes (correct)

✅ **Network Tests Passed:**
- TCP connection to 10.0.0.43:80 successful
- UDP packets sent successfully (542 bytes for 180 LEDs)

✅ **RGB Color Support Confirmed:**
- `IS_GRB_STRIP = False` - WLED expects RGB data
- FrameBuffer sends RGB data (Red-Green-Blue order)
- All effect generators output RGB wire format
- TypeScript code deployed to Android device

### What We Fixed:
- **Protocol Number:** Corrected from 0x01 (WARLS) to 0x02 (DRGB)
- **Color Order:** WLED expects RGB data, not GRB
- **WLED Settings:** Make sure Color Order is set to RGB in Config → LED Preferences

### Expected Results Now:
With `IS_GRB_STRIP = False` (RGB mode):
- RED (255,0,0) → RGB(255,0,0) → Shows RED ✓
- GREEN (0,255,0) → RGB(0,255,0) → Shows GREEN ✓  
- BLUE (0,0,255) → RGB(0,0,255) → Shows BLUE ✓
- WHITE (255,255,255) → RGB(255,255,255) → Shows WHITE ✓

### Next Steps:
1. **Test on Android Device:** The updated app is deployed with RGB support
2. **Run Full Strip Tests:** Try the 180-LED color tests
3. **Test Effect Generators:** Energy bar, pulse, and electric current effects should now work correctly

### WLED Settings to Verify:
- **Settings → Sync Interfaces:** Enable "Receive UDP realtime"
- **Config → LED Preferences → Color Order:** Set to **RGB**
- **Config → LED Preferences → LED Count:** 180